In [ ]:
%%capture
import os
import sys

IN_COLAB = any(k.startswith("COLAB_") for k in os.environ)

if IN_COLAB:
    import re

    import torch

    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {
        "2.10": "0.0.34",
        "2.9": "0.0.33.post1",
        "2.8": "0.0.32.post2",
    }.get(v, "0.0.34")
    %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer pyyaml
    %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    %pip install transformers==4.56.2
    %pip install --no-deps trl==0.22.2
    if not os.path.isdir("/content/belliGPT"):
        !git clone https://github.com/leonardozilli/belliGPT.git /content/belliGPT
    sys.path.insert(0, "/content/belliGPT")
    %env HF_HUB_DISABLE_XET=1
else:
    from pathlib import Path

    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent
    sys.path.insert(0, str(root))

In [ ]:
from huggingface_hub import login

from finetune import env

login(new_session=False)

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

print(env.describe())

In [ ]:
from finetune.config import FinetuneConfig

cfg = FinetuneConfig.compose(
    group_overrides={"model": "minerva3b", "dataset": "sonnets_rhymes"},
).resolve()

print(cfg.summary())

In [ ]:
from finetune import trainer

adapter_dir = trainer.run(cfg)
print("adapter saved to:", adapter_dir)

In [ ]:
from finetune import generate

model, tokenizer = generate.load_adapter(adapter_dir, max_seq_length=cfg.max_seq_length)
print(
    generate.generate(
        model, tokenizer, generate.build_prompt("Li pajacci"), stream=True
    )
)